# Lesson 10 - AI agenti v produkcii

V tejto lekcii sa naučíte **produkčné vzory** pre AI agentov pomocou Microsoft Agent Framework (Python). Pokrývame:

- **Pozorovateľnosť** — pridanie časovania a zaznamenávania do interakcií agenta
- **Vyhodnocovanie** — použitie hodnotiaceho agenta na ohodnotenie kvality odpovedí
- **Správa nákladov** — stratégie pre optimalizáciu tokenov a výber modelu

Scenár je **cestovný agent**, ktorý pomáha používateľom plánovať výlety, s monitorovaním a vyhodnocovaním navrchu.


## Nastavenie


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Produkčné úvahy

Presun AI agentov z prototypov do produkcie vyžaduje starostlivú pozornosť trom pilierom:

1. **Pozorovateľnosť** — Potrebujete vidieť, čo agent robí, ako dlho to trvá a ktoré nástroje volá. Bez trasovania a logovania je ladenie problémov v produkcii takmer nemožné.

2. **Hodnotenie** — Automatizované kontroly kvality zabezpečujú, že odpovede agenta zostávajú presné, úplné a užitočné v priebehu času. Agent hodnotiaci môže skórovať odpovede podľa definovaných kritérií.

3. **Správa nákladov** — Spotreba tokenov priamo ovplyvňuje náklady. Stratégie ako optimalizácia promptov, výber modelu a cachovanie pomáhajú udržiavať výdavky pod kontrolou bez obetovania kvality.


## Vytváranie pozorovateľného agenta

Definujeme cestovné nástroje a obalíme volanie agenta časovaním, aby sme mohli monitorovať oneskorenie. V produkcii by ste integrovali s OpenTelemetry alebo podobným sledovacím backendom.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Vzory hodnotenia

Bežným výrobným vzorom je použitie druhého agenta ako **evaluátora**. Evaluátor hodnotí odpoveď primárneho agenta podľa vopred definovaných kritérií, ako sú úplnosť, presnosť a užitočnosť.

Toto umožňuje:
- Automatizované kontrolné brány kvality predtým, ako odpovede dosiahnu používateľov
- Detekciu regresie pri zmene promptov alebo modelov
- Neustále sledovanie výkonu agenta v priebehu času


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Stratégie riadenia nákladov

Kontrola nákladov je kľúčová pre produkčné AI agenty. Tu sú hlavné stratégie:

| Stratégia | Popis |
|---|---|
| **Optimalizácia promptu** | Udržujte systémové inštrukcie stručné. Odstráňte zbytočný kontext na zníženie počtu vstupných tokenov. |
| **Výber modelu** | Používajte menšie, lacnejšie modely (napr. GPT-4o-mini) pre jednoduché úlohy ako klasifikácia alebo extrakcia a rezervujte väčšie modely pre zložité uvažovanie. |
| **Ukladanie do vyrovnávacej pamäte** | Ukladajte výsledky nástrojov a časté dopyty do vyrovnávacej pamäte, aby ste sa vyhli opakovaným API volaniam. |
| **Rozpočty tokenov** | Nastavte limity `max_tokens`, aby ste predišli neočakávane dlhým odpovediam. |
| **Zoskupovanie (batching)** | Kde je to možné, zoskupujte viacero používateľských dopytov do jedného API volania. |

V praxi funguje dobre viacúrovňový prístup: jednoduché požiadavky smerujte na rýchly, lacný model a zložité dopyty posielajte len na výkonný model.


## Zhrnutie

V tejto lekcii ste sa naučili, ako:

1. **Pridať pozorovateľnosť** do interakcií agenta pomocou časovania a protokolovania, čím sa položili základy pre trasovanie a monitorovanie.
2. **Automaticky vyhodnocovať odpovede agenta** pomocou hodnotiaceho agenta, ktorý skóruje úplnosť, presnosť a užitočnosť.
3. **Riadiť náklady** pomocou optimalizácie promptov, výberu modelu, ukladania do vyrovnávacej pamäte a rozpočtov tokenov.

Tieto produkčné vzory pomáhajú zabezpečiť, aby boli vaše AI agenti spoľahliví, merateľní a nákladovo efektívni vo veľkom meradle.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
